# 03. California CDEC Cooperative Snow Pillow Telemetry (Daily & End-of-Month)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/03_download_cdec_snow_sensors.ipynb)

This notebook discovers all non-SNOTEL California Cooperative Snow Surveys
automated snow pillows on CDEC reporting Sensor `3` (Snow Depth), Sensor `18`
(Snow Water Content), and Sensor `82` (Snow Water Equivalent), downloads their
daily records, and builds **both** Zarr stores ingested by Flowracle:
1. `${FLOWRACLE_DATA_DIR}/daily-ca-snow-sensors/daily_snow_sensors.zarr`
2. `${FLOWRACLE_DATA_DIR}/end-of-month-ca-snow-sensors/endofmonth_snow_sensors.zarr`
   (with a causal $\le 3$-day forward fill prior to month-end resampling).

### Physical Role in Flowracle
In the high southern Sierra Nevada south of $37.0^\circ\text{N}$ (Kings,
Kaweah, Tule, and Kern basins), USDA SNOTEL coverage thins out relative to
California Cooperative Snow Surveys pillows. Flowracle conditionally activates
3D-IDW CDEC `sensor_82` SWE features for watersheds south of `LAT_THRESH = 37.0`.

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0"

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
import io
import os
import shutil
import time
import warnings

from bs4 import BeautifulSoup
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import xarray as xr

warnings.filterwarnings("ignore", category=RuntimeWarning)

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
DAILY_OUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "daily-ca-snow-sensors")
EOM_OUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "end-of-month-ca-snow-sensors")
RAW_WORK_DIR = os.path.join(DAILY_OUT_DIR, "raw_csv")
os.makedirs(RAW_WORK_DIR, exist_ok=True)
os.makedirs(EOM_OUT_DIR, exist_ok=True)

VERIFICATION_MODE = False
SENSORS_TO_DOWNLOAD = [3, 18, 82]
MAX_THREADS = 5

## 1. Discover Non-SNOTEL CDEC Snow Stations & Active Date Ranges

In [ ]:
def get_stations_for_sensor(sensor):
  url = f"https://cdec.water.ca.gov/dynamicapp/staSearch?sta=&sensor_chk=on&sensor={sensor}"
  tables = pd.read_html(url)
  if tables:
    df = tables[0]
    df["Sensor"] = sensor
    return df
  return pd.DataFrame()


def get_sensor_dates(station_id, target_sensors):
  url = f"https://cdec.water.ca.gov/dynamicapp/staMeta?station_id={station_id}"
  results = []
  try:
    resp = requests.get(url, timeout=15)
    soup = BeautifulSoup(resp.content, "html.parser")
    for table in soup.find_all("table"):
      headers = [th.text.strip() for th in table.find_all("th")]
      if "Sensor Number" in headers and "Data Available" in headers:
        for row in table.find_all("tr")[1:]:
          cols = [td.text.strip() for td in row.find_all("td")]
          if len(cols) >= 6:
            try:
              s_num = int(cols[headers.index("Sensor Number")])
            except ValueError:
              continue
            if s_num in target_sensors:
              parts = (
                  cols[headers.index("Data Available")].lower().split(" to ")
              )
              if len(parts) == 2:
                s_dt, e_dt = parts[0].strip(), parts[1].strip()
                if e_dt == "present":
                  e_dt = datetime.now().strftime("%m/%d/%Y")
                results.append({
                    "Station": station_id,
                    "Sensor": s_num,
                    "Start_Date": s_dt,
                    "End_Date": e_dt,
                })
        break
  except Exception as exc:
    print(f"Metadata lookup warning for {station_id}: {exc}")
  return results


all_stations_df = pd.concat(
    [get_stations_for_sensor(s) for s in SENSORS_TO_DOWNLOAD], ignore_index=True
)
snotel_mask = all_stations_df["Operator"].str.contains(
    "Natural Resources Conservation Service|SNOTEL", na=False, case=False
)
all_stations_df = all_stations_df[~snotel_mask]
metadata_file = os.path.join(RAW_WORK_DIR, "station_metadata.csv")
all_stations_df.drop_duplicates(subset=["ID"]).to_csv(
    metadata_file, index=False
)

unique_stations = (
    ["GRZ", "ADM", "MGM"]
    if VERIFICATION_MODE
    else all_stations_df["ID"].unique().tolist()
)
print(
    f"Querying availability windows for {len(unique_stations)} CDEC stations..."
)

inventory = []
with ThreadPoolExecutor(max_workers=MAX_THREADS) as pool:
  futures = [
      pool.submit(get_sensor_dates, stn, SENSORS_TO_DOWNLOAD)
      for stn in unique_stations
  ]
  for fut in as_completed(futures):
    inventory.extend(fut.result())

inventory_df = pd.DataFrame(inventory)
inventory_df["End_Date_DT"] = pd.to_datetime(
    inventory_df["End_Date"], errors="coerce"
)
inventory_df = inventory_df[
    inventory_df["End_Date_DT"] >= pd.Timestamp("1980-01-01")
].drop(columns=["End_Date_DT"])
print(f"Retained {len(inventory_df)} post-1979 station-sensor series.")

## 2. Download Daily CSV Telemetry

In [ ]:
session = requests.Session()
retries = Retry(
    total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504]
)
session.mount("https://", HTTPAdapter(max_retries=retries))

for idx, row in inventory_df.iterrows():
  station, sensor = row["Station"], row["Sensor"]
  start_date, end_date = row["Start_Date"], row["End_Date"]
  if VERIFICATION_MODE:
    end_dt = datetime.strptime(end_date, "%m/%d/%Y")
    start_date = (end_dt - timedelta(days=90)).strftime("%m/%d/%Y")

  sensor_dir = os.path.join(RAW_WORK_DIR, str(sensor))
  os.makedirs(sensor_dir, exist_ok=True)
  url = (
      "https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet"
      f"?Stations={station}&SensorNums={sensor}&dur_code=D&Start={start_date}&End={end_date}"
  )
  try:
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    if "STATION_ID" in resp.text or "DATE TIME" in resp.text:
      pd.read_csv(io.StringIO(resp.text)).to_csv(
          os.path.join(sensor_dir, f"{station}.csv"), index=False
      )
    time.sleep(0.2)
  except Exception as exc:
    print(f"Failed {station} (sensor {sensor}): {exc}")
    time.sleep(1.5)

## 3. Build `daily_snow_sensors.zarr` and `endofmonth_snow_sensors.zarr`

In [ ]:
sensor_arrays = {}
for sensor in SENSORS_TO_DOWNLOAD:
  sensor_dir = os.path.join(RAW_WORK_DIR, str(sensor))
  if not os.path.isdir(sensor_dir):
    continue
  frames = []
  for fname in sorted(os.listdir(sensor_dir)):
    if not fname.endswith(".csv"):
      continue
    df = pd.read_csv(os.path.join(sensor_dir, fname))
    if {"STATION_ID", "DATE TIME", "VALUE"}.issubset(df.columns):
      sub = df[["STATION_ID", "DATE TIME", "VALUE"]].copy()
      sub["VALUE"] = pd.to_numeric(sub["VALUE"], errors="coerce")
      sub["DATE TIME"] = pd.to_datetime(sub["DATE TIME"], errors="coerce")
      frames.append(sub.dropna(subset=["DATE TIME"]))
  if not frames:
    continue
  piv = (
      pd.concat(frames, ignore_index=True)
      .pivot_table(
          index="DATE TIME",
          columns="STATION_ID",
          values="VALUE",
          aggfunc="mean",
      )
      .sort_index()
  )
  sensor_arrays[f"sensor_{sensor}"] = xr.DataArray(
      piv.values,
      coords=[piv.index, piv.columns],
      dims=["time", "station"],
      name=f"sensor_{sensor}",
  )

ds_daily = xr.Dataset(sensor_arrays)
meta_df = (
    pd.read_csv(metadata_file).drop_duplicates(subset=["ID"]).set_index("ID")
)
valid_stns = ds_daily["station"].values
lons_deg_east = (
    meta_df.reindex(valid_stns)["Longitude"].astype(float).values + 360.0
) % 360.0
ds_daily = ds_daily.assign_coords({
    "latitude": (
        "station",
        meta_df.reindex(valid_stns)["Latitude"].astype(float).values,
    ),
    "longitude": ("station", lons_deg_east),
    "elevation": (
        "station",
        meta_df.reindex(valid_stns)["Elevation Feet"].astype(float).values,
    ),
    "operator": (
        "station",
        meta_df.reindex(valid_stns)["Operator"].astype(str).values,
    ),
})
ds_daily.attrs = {
    "description": "California DWR CDEC daily snow sensor data",
    "source": "https://cdec.water.ca.gov/",
}

daily_zarr_path = os.path.join(DAILY_OUT_DIR, "daily_snow_sensors.zarr")
if os.path.exists(daily_zarr_path):
  shutil.rmtree(daily_zarr_path)
ds_daily.to_zarr(daily_zarr_path, mode="w", consolidated=True)
print(f"Wrote daily Zarr: {daily_zarr_path}")

# Causal <=3-day forward fill followed by month-end resampling
filled_das = {
    v: xr.DataArray(
        da.to_pandas().ffill(limit=3),
        dims=da.dims,
        coords=da.coords,
        attrs=da.attrs,
    )
    for v, da in ds_daily.data_vars.items()
}
ds_ffill = xr.Dataset(filled_das, attrs=ds_daily.attrs)
try:
  ds_eom = ds_ffill.resample(time="ME").last()
except ValueError:
  ds_eom = ds_ffill.resample(time="M").last()
ds_eom.attrs["computation_method"] = (
    "End-of-month values derived from daily data using up to 3-day forward"
    " fill."
)
eom_zarr_path = os.path.join(EOM_OUT_DIR, "endofmonth_snow_sensors.zarr")
if os.path.exists(eom_zarr_path):
  shutil.rmtree(eom_zarr_path)
ds_eom.to_zarr(eom_zarr_path, mode="w", consolidated=True)
print(f"Wrote end-of-month Zarr: {eom_zarr_path}")